# Date Cleaning & Seasonality Analysis

----------------------------------------------------------------
## Business Case: Okinsurance — Water-Sport Risk Insurance for US Hotels & Resorts
----------------------------------------------------------------

### This notebook focuses on cleaning and analyzing the `Date` column of the GSAF dataset, in order to test **Hypothesis 2**: shark-incident risk varies significantly by month/season.
### The resulting monthly incident counts support a seasonal premium-loading recommendation for the proposed insurance product.
### Country and Activity cleaning are handled in a separate notebook and merged here for the country + month cross-analysis.

# Date Cleaning & Seasonality Pipeline
### 1. Load dataset
### 2. Explore Date column structure
### 3. Extract Month from parsed dates
### 4. Attempt text-based recovery (regex)
### 5. Combine both approaches
### 6. Investigate and correct the January outlier
### 7. Cross-reference with Country (teammate's cleaning)
### 8. Seasonality result — USA

-------------------------------
##### Dataset:
----------------------------------

#### Global Shark Attack File (GSAF) — https://www.sharkattackfile.net/spreadsheets/GSAF5.xls

#### Importing Libraries: Pandas & Regex

In [15]:
import pandas as pd
import re

-------------------------------------------------------------------

## 1. Load Data

In [16]:
df = pd.read_excel("https://www.sharkattackfile.net/spreadsheets/GSAF5.xls", engine="xlrd")

In [17]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,23rd June,2026.0,Unprovoked,Bahamas,Staniel Cay,Exhuma Cays,Swimming,Unknown,M,12,...,Unknown,Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13th June,2026.0,Unprovoked,USA,Florida,Reservation Way near Indian Pass,Swimming,Keira Ralph,F,17,...,Unknown small shark,Keith Cowley: Simon De Marchi: All things Emer...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13th June,2026.0,Unprovoked,Australia,NSW,Coogee Beach,Swimming,Leah Stewart,F,35,...,Great White Shark 4m,Simon De Marchi: Andrew Currie: ABC NEWS: 9 Ne...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11th June,2026.0,Unprovoked,Galapogos Islands,Santa Fe Island,Mosquera Islet,Snorkeling,Australia Woman,F,?,...,Unknown,Andrew Currie: Facebook: AbsolutCruceros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8th June,2026.0,Unprovoked,USA,Florida,Panama City,Swimming,Unknown,M,20's,...,2.4m (8ft) Bull shark,James Kingsley:Todd Smith: Kevin McMurray Trac...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df.tail()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
7098,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,NaN,...,NaN,"H. Taunton; N. Bartlett, p. 234",ND-0005-RoebuckBay.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0005,ND.0005,6.0,NaN,NaN
7099,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,NaN,Pearl diving,Ahmun,M,NaN,...,NaN,"H. Taunton; N. Bartlett, pp. 233-234",ND-0004-Ahmun.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0004,ND.0004,5.0,NaN,NaN
7100,1900-1905,0.0,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,NaN,...,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0003,ND.0003,4.0,NaN,NaN
7101,1883-1889,0.0,Unprovoked,PANAMA,NaN,"Panama Bay 8ºN, 79ºW",NaN,Jules Patterson,M,NaN,...,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0002,ND.0002,3.0,NaN,NaN
7102,1845-1853,0.0,Unprovoked,CEYLON (SRI LANKA),Eastern Province,"Below the English fort, Trincomalee",Swimming,male,M,15,...,NaN,S.W. Baker,ND-0001-Ceylon.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0001,ND.0001,2.0,NaN,NaN


## 2. Explore Structure: the `Date` column

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7103 entries, 0 to 7102
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7103 non-null   object 
 1   Year            7101 non-null   float64
 2   Type            7085 non-null   object 
 3   Country         7053 non-null   object 
 4   State           6616 non-null   object 
 5   Location        6536 non-null   object 
 6   Activity        6520 non-null   object 
 7   Name            6885 non-null   object 
 8   Sex             6525 non-null   object 
 9   Age             4109 non-null   object 
 10  Injury          7067 non-null   object 
 11  Fatal Y/N       6542 non-null   object 
 12  Time            3576 non-null   object 
 13  Species         3972 non-null   object 
 14  Source          7083 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   object 
 17  href            6796 non-null   o

##### Finding:
##### `Date` is already dtype datetime64[ns] — pandas auto-parsed most values on import.
##### But only 6,146 of 7,103 rows (~86%) have a valid Date.

In [20]:
df['Date'].isna().sum()   # 957 missing (~13.5%)

np.int64(0)

In [21]:
df[df['Date'].isna()][['Date']].head(20)

,Date


##### Finding:
##### The missing rows aren't truly empty in the source file — their original values were free text like "23rd June" (day + month only, no year in the same cell), which pandas couldn't parse automatically.

-------------------------------
## 3. Date Handling — Step 1: Extract Month from parsed dates
-------------------------------

Since `Date` is already parsed for most rows, we can extract `Month` directly, without any extra transformation.

In [22]:
df['Month'] = df['Date'].dt.month

AttributeError: Can only use .dt accessor with datetimelike values

In [1]:
df['Month'].isna().sum()   # 957 rows -> will be excluded from the seasonality analysis

NameError: name 'df' is not defined

In [2]:
df['Month'].isna().mean() * 100   # ~13.5% of rows lack a usable month

NameError: name 'df' is not defined

In [3]:
df.dropna(subset=['Month'])   # demonstrates the null-handling technique
# (a filtered working copy, df_month_clean, is built later in Step 6, after the outlier correction)

NameError: name 'df' is not defined

-------------------------------
## 4. Date Handling — Step 2: Attempting text-based recovery (regex)
-------------------------------

For the ~957 rows with a missing `Date`, we suspected the raw text was still present in the source file but had been lost during pandas' automatic type conversion. To test this, we re-read the file forcing `Date` to stay as plain text, then used regex to extract the last word of each string (expected to be the month name, e.g. "June" from "23rd June").

In [ ]:
df_dates_fixed = pd.read_excel("https://www.sharkattackfile.net/spreadsheets/GSAF5.xls", engine="xlrd", dtype={'Date': str})

In [ ]:
df_dates_fixed['Date'].head(10)

In [ ]:
df_dates_fixed['Month_name'] = df_dates_fixed['Date'].str.extract(r'([A-Za-z]+)$')
# regex captures only the last word of the string (the month name), ignoring day numbers
# and suffixes like rd/st/th/nd

In [ ]:
df_dates_fixed['Month'] = pd.to_datetime(df_dates_fixed['Month_name'], format='%B', errors='coerce').dt.month

In [ ]:
df_dates_fixed['Month_name'].value_counts()
# reveals inconsistent formatting: some rows aren't "day + month" text at all

In [ ]:
df_dates_fixed['Month'].notna().sum()   # only 74 valid months recovered this way

##### Finding:
##### This approach alone only recovered 74 valid months across the whole dataset — far fewer than expected — because it ignores the ~86% of dates pandas had already parsed correctly on the first read. It also revealed genuinely inconsistent formatting in the source column (e.g. "Kong", "weekend", "Ladysmith" were extracted instead of month names).

-------------------------------
## 5. Date Handling — Step 3: Combined Approach
-------------------------------

We combine both methods: keep the `Month` already extracted from the correctly auto-parsed dates (Step 1), and only apply the regex recovery (Step 2) to the rows that were still missing.

In [ ]:
# extracts the month directly from dates that were already parsed correctly
df['Month'] = df['Date'].dt.month

# for the remaining missing rows, recover the month from the raw text version (regex)
df_raw_dates = pd.read_excel("https://www.sharkattackfile.net/spreadsheets/GSAF5.xls", engine="xlrd", dtype={'Date': str})

rows_without_month = df['Month'].isna()   # identifies which rows are still missing a month
month_text = df_raw_dates.loc[rows_without_month, 'Date'].str.extract(r'([A-Za-z]+)$')[0]   # regex recovery
month_recovered = pd.to_datetime(month_text, format='%B', errors='coerce').dt.month   # text -> number (e.g. "May" -> 5)

df.loc[rows_without_month, 'Month'] = month_recovered   # fills in only the rows that were missing

In [ ]:
df['Month'].isna().sum()   # missing rows after the combined approach

In [ ]:
df['Month'].isna().mean() * 100   # ~12.4% missing rate (down from 13.5%)

In [ ]:
df['Month'].value_counts().sort_index()   # monthly incident counts (before outlier correction)

##### Finding:
##### The combined approach reduced the missing-value rate from 13.5% to ~12.4%.

-------------------------------
## 6. Outlier Investigation — the January Anomaly
-------------------------------

January shows an implausible 812 incidents in the table above — nearly double every neighboring month. This is worth investigating before trusting it.

In [ ]:
jan_rows = df['Month'] == 1
(df.loc[jan_rows, 'Date'].dt.day == 1).sum()   # how many January rows fall exactly on day 1

##### Finding:
##### 334 of the 812 January rows (41%) have the exact date "January 1st" — a signature of incomplete original dates (year only, e.g. "1921"), which Excel/pandas defaults to January 1st when parsing. This is a structural artifact, not a real seasonal spike.

In [ ]:
# excludes rows where an incomplete date was defaulted to "January 1st"
january_incomplete_dates = (df['Month'] == 1) & (df['Date'].dt.day == 1)
df_month_clean = df.loc[~january_incomplete_dates].copy()

In [ ]:
# final, corrected incident count by month
incidents_by_month = df_month_clean['Month'].value_counts().sort_index()
incidents_by_month

##### Finding:
##### January drops from 812 to a much more realistic 478 once the outlier is removed — in line with neighboring months.

-------------------------------
## 7. Cross-Referencing with Country
-------------------------------

Country cleaning was done separately (see teammate's notebook). We reuse that same cleaning logic here to cross `Month` with `Country`.

In [ ]:
# creates a cleaned version of Country (same cleaning logic as the teammate's notebook, applied here to cross-reference with Month)
df2 = df.copy()

df2["Country"] = (
    df2["Country"]
    .str.strip()
    .str.title()
    .str.replace(r"\?", "", regex=True)
    .str.replace(r"[^\w\s]", "", regex=True)
)

region_list = ["Africa", "Asia", "Red Sea"]
df2["Country"] = df2["Country"].replace(region_list, "Unknown")
df2["Country"] = df2["Country"].replace(r"^Between.*", "Unknown", regex=True)

In [ ]:
df2['Country'].head()

In [ ]:
df_month_clean['Country'] = df2['Country']

In [ ]:
country_month_summary = (
    df_month_clean
    .groupby(['Country', 'Month'])
    .size()
    .reset_index(name='Incidents')
)
country_month_summary.head(20)

-------------------------------
## 8. Seasonality Result — USA
-------------------------------

Since the proposed insurance product targets US hotels & resorts, we narrow the analysis to the USA specifically.

In [ ]:
usa_summary = country_month_summary[country_month_summary['Country'] == 'Usa']
usa_summary.sort_values('Month')

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(usa_summary['Month'], usa_summary['Incidents'])
plt.title("Monthly Shark Incidents in the USA")
plt.xlabel("Month")
plt.ylabel("Incidents")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

##### Note: incidents peak sharply in July, nearly double the surrounding months — directly supporting a seasonal premium adjustment for peak summer months (Hypothesis 2 supported).